# Week 6 — Lab
## Grad-CAM, attention rollout, and faithfulness on real images

End-to-end:

1. Load a small set of images and a pretrained **ResNet-50**. Run **Grad-CAM** and
   **Grad-CAM++** at the last conv stage.
2. Load a pretrained **ViT-Base/16** from `timm` and run **attention rollout**.
3. Build a **deletion-AUC** evaluation that scores each method's faithfulness, not just
   its visual plausibility.
4. Run a **parameter-randomization sanity check** on Grad-CAM.

The lab assumes you have `pytorch-grad-cam` and `timm` installed (see the course
`requirements.txt`). It runs on CPU in a few minutes; a GPU speeds it up but is not
required.


In [ ]:
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
import torchvision
from torchvision import transforms as T
from PIL import Image
import matplotlib.pyplot as plt

plt.style.use("../../assets/mplstyle/course.mplstyle")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DATA = Path("../data")
IMG_PATHS = sorted(DATA.glob("*.jpg"))
print("images:", [p.name for p in IMG_PATHS])


## 1. Grad-CAM on ResNet-50

We use the maintained `pytorch-grad-cam` package. The target layer is the last bottleneck
block (`layer4[-1]`).

In [ ]:
from pytorch_grad_cam import GradCAM, GradCAMPlusPlus, EigenCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

weights = torchvision.models.ResNet50_Weights.DEFAULT
model = torchvision.models.resnet50(weights=weights).to(DEVICE).eval()
preprocess = weights.transforms()
categories = weights.meta["categories"]


def load_tensor(path):
    img = Image.open(path).convert("RGB").resize((224, 224))
    rgb = np.asarray(img).astype(np.float32) / 255.0
    x = preprocess(img).unsqueeze(0).to(DEVICE)
    return rgb, x


def topk(logits, k=3):
    probs = F.softmax(logits, dim=-1)[0]
    idx = torch.topk(probs, k).indices.cpu().tolist()
    return [(categories[i], probs[i].item()) for i in idx]


# Sanity check — what does the model think these images are?
for p in IMG_PATHS:
    _, x = load_tensor(p)
    with torch.no_grad():
        logits = model(x)
    print(f"{p.name:12s}  top-3: ", [(c, f'{p_:.2f}') for c, p_ in topk(logits)])


**Note.** The synthetic images bundled with the course are deliberately
non-photorealistic, so ResNet-50's top-1 may not match the filename. That's fine — the
attribution methods explain whatever class we *ask* for, not the class we think the
image looks like. Drop your own photos into `data/` to see the methods on natural
images.

### Grad-CAM and Grad-CAM++ side by side

In [ ]:
def run_cam(cam_cls, target_layer, x, target_class):
    cam = cam_cls(model=model, target_layers=[target_layer])
    return cam(input_tensor=x,
               targets=[ClassifierOutputTarget(target_class)])[0]  # H×W in [0, 1]


target_layer = model.layer4[-1]

fig, axes = plt.subplots(len(IMG_PATHS), 3, figsize=(8, 2.6 * len(IMG_PATHS)))
for row, p in enumerate(IMG_PATHS):
    rgb, x = load_tensor(p)
    with torch.no_grad():
        pred = int(model(x).argmax(-1))
    cam_a = run_cam(GradCAM,        target_layer, x, pred)
    cam_b = run_cam(GradCAMPlusPlus, target_layer, x, pred)

    axes[row, 0].imshow(rgb); axes[row, 0].axis("off")
    axes[row, 0].set_title(f"{p.stem} → {categories[pred][:18]}", fontsize=9)
    axes[row, 1].imshow(show_cam_on_image(rgb, cam_a, use_rgb=True)); axes[row, 1].axis("off")
    axes[row, 2].imshow(show_cam_on_image(rgb, cam_b, use_rgb=True)); axes[row, 2].axis("off")
    if row == 0:
        axes[row, 1].set_title("Grad-CAM"); axes[row, 2].set_title("Grad-CAM++")
plt.tight_layout(); plt.show()


**Reading.** The heatmap concentrates on the region the network used to assign
the top class. Grad-CAM++ and Grad-CAM are usually close on single-object images;
Grad-CAM++ pulls ahead when the class appears in multiple regions.

### EigenCAM — the class-agnostic baseline

EigenCAM computes the leading principal component of the feature map tensor at the
chosen layer — *without* using the class score. If your Grad-CAM is roughly the same
shape as EigenCAM, the Grad-CAM is not really discriminating between classes.

In [ ]:
fig, axes = plt.subplots(1, len(IMG_PATHS), figsize=(2.6 * len(IMG_PATHS), 2.8))
for ax, p in zip(axes, IMG_PATHS):
    rgb, x = load_tensor(p)
    with torch.no_grad():
        pred = int(model(x).argmax(-1))
    cam = run_cam(EigenCAM, target_layer, x, pred)
    ax.imshow(show_cam_on_image(rgb, cam, use_rgb=True))
    ax.set_title(p.stem); ax.axis("off")
fig.suptitle("EigenCAM (class-agnostic) — sanity baseline")
plt.tight_layout(); plt.show()


## 2. Attention rollout on a ViT

We use `timm`'s `vit_base_patch16_224` (pretrained on ImageNet) and roll out attention
across all 12 layers.

The implementation is short enough to write by hand — it makes the algorithm
concrete.

In [ ]:
import timm

vit = timm.create_model("vit_base_patch16_224", pretrained=True).to(DEVICE).eval()
vit_transform = T.Compose([T.Resize(224), T.CenterCrop(224), T.ToTensor(),
                           T.Normalize(mean=[0.5]*3, std=[0.5]*3)])

# Hook every attention block to capture its attention weights
attn_maps = []
def hook(module, input, output):
    # timm Attention modules don't return weights by default; we patch the forward.
    pass

# Easier path: monkey-patch each block to store attention
orig_forwards = []
def patched_forward_factory(block):
    attn = block.attn
    orig = attn.forward
    def fwd(x, *args, **kwargs):
        B, N, C = x.shape
        qkv = attn.qkv(x).reshape(B, N, 3, attn.num_heads, attn.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        q = q * attn.scale
        attn_mat = (q @ k.transpose(-2, -1)).softmax(dim=-1)
        attn_maps.append(attn_mat.detach())            # (B, H, N, N)
        attn_drop = attn.attn_drop(attn_mat)
        x = (attn_drop @ v).transpose(1, 2).reshape(B, N, C)
        x = attn.proj_drop(attn.proj(x))
        return x
    return fwd

for block in vit.blocks:
    orig_forwards.append(block.attn.forward)
    block.attn.forward = patched_forward_factory(block)


In [ ]:
def attention_rollout(image_path):
    attn_maps.clear()
    img = Image.open(image_path).convert("RGB")
    x = vit_transform(img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        logits = vit(x)
    pred = int(logits.argmax(-1))

    # Average over heads, add identity for the residual, normalize rows
    rolled = None
    for A in attn_maps:                            # each (1, H, N, N)
        a = A.mean(dim=1)[0]                       # average heads -> (N, N)
        a = a + torch.eye(a.size(0), device=a.device)
        a = a / a.sum(dim=-1, keepdim=True)
        rolled = a if rolled is None else a @ rolled

    # The CLS token is the first token; its row tells us what it attended to
    mask = rolled[0, 1:].cpu().numpy()             # drop the CLS-to-CLS entry
    side = int(np.sqrt(mask.size))
    mask = mask.reshape(side, side)
    mask = (mask - mask.min()) / (mask.max() - mask.min() + 1e-9)
    return img.resize((224, 224)), mask, pred


fig, axes = plt.subplots(1, len(IMG_PATHS), figsize=(2.8 * len(IMG_PATHS), 3))
for ax, p in zip(axes, IMG_PATHS):
    rgb, mask, pred = attention_rollout(p)
    rgb_arr = np.asarray(rgb).astype(np.float32) / 255.0
    mask_up = np.array(Image.fromarray((mask * 255).astype(np.uint8)).resize((224, 224))) / 255.0
    overlay = (1 - mask_up[..., None]) * 0.4 * rgb_arr + mask_up[..., None] * np.array([1, 0, 0])
    ax.imshow(np.clip(0.5 * rgb_arr + 0.5 * overlay, 0, 1))
    ax.set_title(f"{p.stem}", fontsize=9); ax.axis("off")
fig.suptitle("ViT-B/16 attention rollout (red = high attention from [CLS])")
plt.tight_layout(); plt.show()

# Restore original forwards
for block, fwd in zip(vit.blocks, orig_forwards):
    block.attn.forward = fwd


**Reading.** Attention rollout concentrates on the regions the `[CLS]` token
attended to — often the object centre and a few outline patches. It is **less
class-discriminative** than Grad-CAM, by design: any token's attention is shared across
many tasks the model is computing in parallel.

## 3. Faithfulness — deletion AUC

We score Grad-CAM with the **deletion AUC**: starting from the original image, we
delete (set to the mean pixel) the top-K % of pixels by the attribution score and watch
the class probability fall. A steep fall = faithful explanation.

In [ ]:
def deletion_curve(image_path, n_steps=20):
    rgb, x = load_tensor(image_path)
    with torch.no_grad():
        pred = int(model(x).argmax(-1))
        p0 = F.softmax(model(x), dim=-1)[0, pred].item()
    cam = run_cam(GradCAM, target_layer, x, pred)        # H×W in [0,1]

    flat = cam.flatten()
    order = np.argsort(flat)[::-1]                       # highest first

    cur = x.clone()
    mean_pix = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
    fracs = np.linspace(0, 1, n_steps + 1)
    ps = []
    for f in fracs:
        cur = x.clone()
        n = int(f * len(order))
        if n > 0:
            idx_h, idx_w = np.unravel_index(order[:n], cam.shape)
            # apply to the normalized tensor space — use a per-channel mean target
            for c in range(3):
                cur[0, c, idx_h, idx_w] = ((mean_pix[0, c] - weights.transforms().mean[c])
                                            / weights.transforms().std[c])
        with torch.no_grad():
            ps.append(F.softmax(model(cur), dim=-1)[0, pred].item())
    return fracs, ps, p0, pred


fig, ax = plt.subplots(figsize=(7, 4))
all_aucs = []
for p in IMG_PATHS:
    fr, ps, _, pred = deletion_curve(p)
    auc = np.trapz(ps, fr)
    all_aucs.append(auc)
    ax.plot(fr, ps, label=f"{p.stem} ({categories[pred][:14]})  AUC={auc:.3f}")
ax.set(xlabel="fraction of top-CAM pixels deleted",
       ylabel="class probability",
       title="Deletion curves for Grad-CAM\n(lower = more faithful)")
ax.legend(loc="upper right", fontsize=8)
plt.show()

print(f"\nMean deletion AUC across {len(IMG_PATHS)} images: {np.mean(all_aucs):.3f}")


**Reading.** A steeply-dropping curve says Grad-CAM correctly identified the
pixels the model relied on — removing them collapses the probability. A shallow curve
says either (a) the model is robust to occlusion of the highlighted region (and the
attribution missed the real driver), or (b) the model has redundant features (the
explanation is true but incomplete).

For a paper, compute deletion AUC over a held-out test set of size ≥ 200 and report
the mean and standard deviation. **That number is the difference between "we made a
nice picture" and "we evaluated the explanation".**

## 4. Sanity check: random-init Grad-CAM

The Adebayo et al. sanity check: rerun Grad-CAM on a randomly-initialized copy of the
same architecture. The maps should be obviously different.

In [ ]:
random_model = torchvision.models.resnet50(weights=None).to(DEVICE).eval()
random_target_layer = random_model.layer4[-1]


def run_cam_on(model_, layer_, x, target_class):
    cam = GradCAM(model=model_, target_layers=[layer_])
    return cam(input_tensor=x,
               targets=[ClassifierOutputTarget(target_class)])[0]


fig, axes = plt.subplots(len(IMG_PATHS), 3, figsize=(8, 2.5 * len(IMG_PATHS)))
for row, p in enumerate(IMG_PATHS):
    rgb, x = load_tensor(p)
    with torch.no_grad():
        pred = int(model(x).argmax(-1))
        pred_rand = int(random_model(x).argmax(-1))
    cam_trained = run_cam_on(model, target_layer, x, pred)
    cam_random  = run_cam_on(random_model, random_target_layer, x, pred_rand)

    axes[row, 0].imshow(rgb); axes[row, 0].axis("off")
    axes[row, 0].set_title(p.stem, fontsize=9)
    axes[row, 1].imshow(show_cam_on_image(rgb, cam_trained, use_rgb=True)); axes[row, 1].axis("off")
    axes[row, 2].imshow(show_cam_on_image(rgb, cam_random,  use_rgb=True)); axes[row, 2].axis("off")
    if row == 0:
        axes[row, 1].set_title("trained"); axes[row, 2].set_title("random init")
fig.suptitle("Grad-CAM sanity check — trained vs. randomly initialised")
plt.tight_layout(); plt.show()


**Reading.** Trained-model Grad-CAM concentrates on the object; random-model
Grad-CAM is approximately uniform noise. **They are obviously different**, which is
the outcome the sanity check is designed to detect. Grad-CAM passes.

## What to do differently in your own research

- For CNNs, default to **Grad-CAM++** on the last convolutional layer.
- For ViTs, default to **attention rollout**, but **do not** stop at the visualization
  — pair it with a faithfulness score.
- Always report a **quantitative** faithfulness metric (deletion or insertion AUC)
  averaged over a held-out set. The map is a hypothesis; the AUC is the test.
- Run **at least one** sanity check (random-init or random-label) and report the
  outcome. A method that fails sanity checks should be reported as a heatmap of where
  the model **looked**, not where the model **decided**.

### Next

Open `exercises/01-faithfulness.ipynb` for two exercises on choosing the right target
layer and on the insertion-AUC counterpart of the deletion metric.
